# 실험 02: Tool 실행 사이클 완성하기 (ToolMessage)

## 1. 실험 제목과 목표
- **제목**: LLM 대신 계산기 쳐주기
- **목표**: 교안 12번 슬라이드에 명시된 "LLM 판단 -> Tool 입력값 생성 -> **애플리케이션이 Tool 실행** -> **Tool 결과 반환** -> LLM 요약"의 6단계 풀-사이클(Full-cycle)을 파이썬 코드로 구현합니다.

## 2. 실험 개요
1. **실험 1 (수동 실행 사이클)**: LLM이 뱉은 `tool_calls`에서 파라미터를 꺼내 진짜 함수에 넣고 돌린 뒤, 그 결과를 `ToolMessage` 객체에 담아 다시 LLM에게 던지는 구조를 실습합니다.
2. **실패/주의 케이스**: Tool 실행 중에 파이썬 에러(예: 0으로 나누기)가 발생했을 때, 프로그램이 죽지 않고 LLM에게 에러 메시지를 전달하여 LLM이 자연스럽게 사과하도록 만드는 에러 핸들링 기법

## 3. 라이브러리 import

In [1]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

## 4. 실험 1: ToolMessage를 활용한 전체 사이클 조립
나눗셈 기능을 하는 간단한 툴을 만들고, 전체 대화 맥락(History)을 유지하며 통신하는 과정을 구현합니다.

In [2]:
@tool
def divide_numbers(a: float, b: float) -> float:
    """a를 b로 나누는 계산기입니다."""
    return a / b

llm_with_tools = llm.bind_tools([divide_numbers])

print("=== [1. Human -> LLM (질문)] ===")
# LangChain에서는 여러 턴의 대화를 주고받을 때 리스트 형태의 Message History를 사용합니다.
messages = [HumanMessage(content="10만 원을 3명이서 똑같이 나누면 얼마야?")]

# 1차 호출: LLM이 툴 호출을 판단
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg) # 대화 기록에 LLM의 응답(tool_calls 포함) 추가
print("LLM 응답:", ai_msg.tool_calls)

print("\n=== [2. Application -> Tool (실행)] ===")
# 툴 실행 로직
for tool_call in ai_msg.tool_calls:
    if tool_call["name"] == "divide_numbers":
        print(f"-> 파이썬 코드에서 divide_numbers 실행 중... 인자: {tool_call['args']}")
        # 딕셔너리로 된 args를 **kwargs 형태로 함수에 쏴줍니다.
        result = divide_numbers.invoke(tool_call["args"])
        print(f"-> 파이썬 실행 결과: {result}")
        
        # ***핵심***: 툴 실행 결과를 ToolMessage에 담습니다. (어떤 툴을 실행한 결과인지 id 매칭 필수)
        tool_msg = ToolMessage(
            content=str(result), 
            tool_call_id=tool_call["id"]
        )
        messages.append(tool_msg) # 대화 기록에 툴 결과 추가

print("\n=== [3. Application -> LLM (최종 요약 요청)] ===")
# 2차 호출: 툴 결과를 받은 LLM이 최종 자연어 답변을 생성
final_res = llm_with_tools.invoke(messages)
print("최종 답변:", final_res.content)

=== [1. Human -> LLM (질문)] ===
LLM 응답: [{'name': 'divide_numbers', 'args': {'a': 100000, 'b': 3}, 'id': 'call_CWWooFsG6EvTPVvRz6p4EVt3', 'type': 'tool_call'}]

=== [2. Application -> Tool (실행)] ===
-> 파이썬 코드에서 divide_numbers 실행 중... 인자: {'a': 100000, 'b': 3}
-> 파이썬 실행 결과: 33333.333333333336

=== [3. Application -> LLM (최종 요약 요청)] ===
최종 답변: 10만 원을 3명이서 똑같이 나누면 각자 33,333.33원이 됩니다.


## 5. 실패/주의 케이스: 0으로 나누기 (에러 핸들링)
LLM이 `b`에 0을 넣으라고 명령하면 파이썬은 `ZeroDivisionError`를 뿜으며 죽습니다. 이를 우아하게 처리해 봅시다.

In [3]:
messages_bad = [HumanMessage(content="10을 0으로 나누면 얼마야?")]
ai_msg_bad = llm_with_tools.invoke(messages_bad)
messages_bad.append(ai_msg_bad)

print("[파이썬 실행 중 에러 발생 시뮬레이션]")
for tool_call in ai_msg_bad.tool_calls:
    try:
        # 여기서 ZeroDivisionError가 팍!
        result = divide_numbers.invoke(tool_call["args"])
        tool_content = str(result)
    except Exception as e:
        print("🔥 에러 포착! 서버가 죽지 않게 잡았습니다:", e)
        # 에러 메시지 자체를 문자열로 만들어서 LLM에게 줍니다.
        tool_content = f"Error occurred: {str(e)}"
        
    messages_bad.append(ToolMessage(content=tool_content, tool_call_id=tool_call["id"]))

print("\n[에러를 전달받은 LLM의 최종 답변]")
final_bad_res = llm_with_tools.invoke(messages_bad)
print(final_bad_res.content)
print("\n-> 결과: 서버가 죽는 대신, LLM이 '0으로 나눌 수 없다는 에러가 났다'는 걸 인지하고 사용자에게 자연스럽게 이유를 설명합니다. 이것이 Tool Calling 에러 핸들링의 정석입니다.")

[파이썬 실행 중 에러 발생 시뮬레이션]
🔥 에러 포착! 서버가 죽지 않게 잡았습니다: float division by zero

[에러를 전달받은 LLM의 최종 답변]
0으로 나누는 것은 수학적으로 정의되지 않아서 오류가 발생합니다. 따라서 10을 0으로 나누는 것은 불가능합니다.

-> 결과: 서버가 죽는 대신, LLM이 '0으로 나눌 수 없다는 에러가 났다'는 걸 인지하고 사용자에게 자연스럽게 이유를 설명합니다. 이것이 Tool Calling 에러 핸들링의 정석입니다.


## 6. 결과 해석

1. **메시지 체인(History)**: Tool Calling은 한 번의 요청/응답으로 끝나지 않습니다. `[Human -> AI(tool_calls) -> Tool(결과) -> AI(최종답변)]` 이라는 핑퐁 릴레이가 필요하며, 이 내역을 리스트(`messages`)에 누적해서 계속 넘겨줘야 합니다.
2. **Agent로의 진화**: 지금은 `for` 문을 돌려서 우리가 수동으로 툴을 실행해 줬지만, 툴이 수십 개가 되면 이걸 일일이 짤 수 없습니다. 이 **for문 루프를 알아서 돌아주는 시스템**이 바로 다음 챕터에서 배울 **LangGraph (또는 AgentExecutor)** 입니다.
3. **안정성 확보**: `try-except`로 툴 에러를 잡아내 LLM에게 넘기는 패턴은 실제 AI 서비스 구축 시 서비스 장애를 막는 가장 중요한 생명줄입니다.

## 7. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* Tool Calling 사이클을 수동으로 구현해 보니, 핑퐁을 2번 쳐야 한다는 것을 확실히 이해함.
* 에러가 났을 때 에러 코드를 그냥 LLM한테 던져주면, LLM이 찰떡같이 알아듣고 사용자에게 친절하게 통역해 준다는 게 매우 놀라움.

### 다음 개선 방향
* 교안 슬라이드에 나온 것처럼, 툴 하나만 쓰는 게 아니라 계산기, 검색기, 문서 검색기 등 3~4개의 툴을 한 번에 주면 LLM이 알아서 적절한 툴을 고르는(Routing) 마법을 시연해 볼 차례임.